# 02 – Imunisasi Campak Balita per Provinsi 2025 (D2)
Bagian D2 dalam proyek dashboard SDG 3 (Campak di Indonesia):
* Menjawab pertanyaan: **"Apakah imunisasi merata atau timpang antarprovinsi, dan di mana celah terbesarnya?"**
* Menghasilkan:
  1. `provinsi_2025.csv`: Tabel bersih 38 provinsi untuk Looker Studio (peta & bar chart).
  2. `tabel_padanan_provinsi.csv`: Padanan nama provinsi (BPS ↔ GeoJSON ↔ Kode Wilayah).
  3. Tiga kalimat temuan utama untuk Bab III & Bab IV Makalah serta PPT.


## 1. Pengaturan Lingkungan


In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

# Struktur repo: masukan dibaca dari data/raw/, keluaran ditulis ke data/clean/.
# Notebook bisa dijalankan dari folder notebooks/ atau dari root repo.
ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
RAW = ROOT / "data" / "raw"
CLEAN = ROOT / "data" / "clean"
CLEAN.mkdir(parents=True, exist_ok=True)

FILE_BPS = RAW / "persentase-imunisasi-campak-balita-provinsi-2025.csv"
FILE_GEOJSON = RAW / "indonesia-geojson-topojson-maps-with-38-provinces-main/indonesia-geojson-topojson-maps-with-38-provinces-main/GeoJSON/indonesia-38-provinces.geojson"

kurang = [f for f in [FILE_BPS, FILE_GEOJSON] if not f.exists()]
assert not kurang, f"File masukan belum ada: {kurang}"
print("Membaca data mentah dari :", RAW)
print("Menyimpan data bersih ke :", CLEAN)

Membaca data mentah dari : ..\data\raw
Menyimpan data bersih ke : ..\data\clean


## 2. Membaca & Membersihkan Data BPS 2025
File BPS `persentase-imunisasi-campak-balita-provinsi-2025.csv` memiliki 3 baris judul di bagian atas dan 1 baris total nasional (`INDONESIA`).
Tahapan:
1. Membaca data dan memisahkan angka rata-rata nasional.
2. Mengambil 38 data provinsi.


In [2]:
file_bps = FILE_BPS

with open(file_bps, "r", encoding="utf-8") as f:
    baris = [b.strip() for b in f if b.strip()]

# Baris 0–2 adalah judul tabel, baris 3 ke bawah adalah data
data_provinsi = []
angka_nasional = None

for b in baris[3:]:
    kolom = b.split(",")
    nama = kolom[0].strip()
    nilai = float(kolom[1].strip())
    if nama == "INDONESIA":
        angka_nasional = nilai
    else:
        data_provinsi.append({"nama_bps": nama, "imunisasi_campak_pct": nilai})

df_bps = pd.DataFrame(data_provinsi)
print(f"Data BPS berhasil dibaca: {len(df_bps)} provinsi.")
print(f"Rata-rata Nasional Indonesia: {angka_nasional}%")
display(df_bps.head())


Data BPS berhasil dibaca: 38 provinsi.
Rata-rata Nasional Indonesia: 63.52%


,nama_bps,imunisasi_campak_pct
0,ACEH,27.79
1,SUMATERA UTARA,53.25
2,SUMATERA BARAT,43.50
3,RIAU,51.78
4,JAMBI,59.16


## 3. Membaca GeoJSON 38 Provinsi & Membuat Tabel Padanan
Terdapat 3 nama provinsi yang ditulis berbeda antara BPS dan GeoJSON:
* `DI YOGYAKARTA` (BPS) ↔ `Daerah Istimewa Yogyakarta` (GeoJSON)
* `KEP. BANGKA BELITUNG` (BPS) ↔ `Kepulauan Bangka Belitung` (GeoJSON)
* `KEP. RIAU` (BPS) ↔ `Kepulauan Riau` (GeoJSON)

Selain itu, 4 provinsi pemekaran baru di Papua (Papua Selatan, Tengah, Pegunungan, Barat Daya) memiliki kode spesifik pada peta. Kita susun tabel padanan resmi agar penggabungan di Looker Studio tidak bermasalah.


In [3]:
file_geojson = FILE_GEOJSON

with open(file_geojson, "r", encoding="utf-8") as f:
    geo_data = json.load(f)

# Kamus pemetaan BPS ke nama standar, kode provinsi resmi, dan nama di GeoJSON
bps_to_meta = {
    'ACEH': ('11', 'Aceh', 'Aceh'),
    'SUMATERA UTARA': ('12', 'Sumatera Utara', 'Sumatera Utara'),
    'SUMATERA BARAT': ('13', 'Sumatera Barat', 'Sumatera Barat'),
    'RIAU': ('14', 'Riau', 'Riau'),
    'JAMBI': ('15', 'Jambi', 'Jambi'),
    'SUMATERA SELATAN': ('16', 'Sumatera Selatan', 'Sumatera Selatan'),
    'BENGKULU': ('17', 'Bengkulu', 'Bengkulu'),
    'LAMPUNG': ('18', 'Lampung', 'Lampung'),
    'KEP. BANGKA BELITUNG': ('19', 'Kepulauan Bangka Belitung', 'Kepulauan Bangka Belitung'),
    'KEP. RIAU': ('21', 'Kepulauan Riau', 'Kepulauan Riau'),
    'DKI JAKARTA': ('31', 'DKI Jakarta', 'DKI Jakarta'),
    'JAWA BARAT': ('32', 'Jawa Barat', 'Jawa Barat'),
    'JAWA TENGAH': ('33', 'Jawa Tengah', 'Jawa Tengah'),
    'DI YOGYAKARTA': ('34', 'DI Yogyakarta', 'Daerah Istimewa Yogyakarta'),
    'JAWA TIMUR': ('35', 'Jawa Timur', 'Jawa Timur'),
    'BANTEN': ('36', 'Banten', 'Banten'),
    'BALI': ('51', 'Bali', 'Bali'),
    'NUSA TENGGARA BARAT': ('52', 'Nusa Tenggara Barat', 'Nusa Tenggara Barat'),
    'NUSA TENGGARA TIMUR': ('53', 'Nusa Tenggara Timur', 'Nusa Tenggara Timur'),
    'KALIMANTAN BARAT': ('61', 'Kalimantan Barat', 'Kalimantan Barat'),
    'KALIMANTAN TENGAH': ('62', 'Kalimantan Tengah', 'Kalimantan Tengah'),
    'KALIMANTAN SELATAN': ('63', 'Kalimantan Selatan', 'Kalimantan Selatan'),
    'KALIMANTAN TIMUR': ('64', 'Kalimantan Timur', 'Kalimantan Timur'),
    'KALIMANTAN UTARA': ('65', 'Kalimantan Utara', 'Kalimantan Utara'),
    'SULAWESI UTARA': ('71', 'Sulawesi Utara', 'Sulawesi Utara'),
    'SULAWESI TENGAH': ('72', 'Sulawesi Tengah', 'Sulawesi Tengah'),
    'SULAWESI SELATAN': ('73', 'Sulawesi Selatan', 'Sulawesi Selatan'),
    'SULAWESI TENGGARA': ('74', 'Sulawesi Tenggara', 'Sulawesi Tenggara'),
    'GORONTALO': ('75', 'Gorontalo', 'Gorontalo'),
    'SULAWESI BARAT': ('76', 'Sulawesi Barat', 'Sulawesi Barat'),
    'MALUKU': ('81', 'Maluku', 'Maluku'),
    'MALUKU UTARA': ('82', 'Maluku Utara', 'Maluku Utara'),
    'PAPUA BARAT': ('92', 'Papua Barat', 'Papua Barat'),
    'PAPUA BARAT DAYA': ('96', 'Papua Barat Daya', 'Papua Barat Daya'),
    'PAPUA': ('91', 'Papua', 'Papua'),
    'PAPUA SELATAN': ('93', 'Papua Selatan', 'Papua Selatan'),
    'PAPUA TENGAH': ('94', 'Papua Tengah', 'Papua Tengah'),
    'PAPUA PEGUNUNGAN': ('95', 'Papua Pegunungan', 'Papua Pegunungan'),
}

geo_dict = {feat['properties']['PROVINSI']: feat['properties'] for feat in geo_data['features']}

padanan_rows = []
for nama_b, (k_prov, nama_std, nama_g) in bps_to_meta.items():
    g_prop = geo_dict[nama_g]
    padanan_rows.append({
        'kode_prov': k_prov,
        'nama_bps': nama_b,
        'provinsi': nama_std,
        'nama_peta': nama_g,
        'geojson_id': g_prop['id'],
        'geojson_kode_prov': g_prop['KODE_PROV']
    })

df_padanan = pd.DataFrame(padanan_rows).sort_values('kode_prov').reset_index(drop=True)
print(f"Tabel padanan selesai: {len(df_padanan)} provinsi terpetakan 100% ke GeoJSON.")
display(df_padanan.head(5))


Tabel padanan selesai: 38 provinsi terpetakan 100% ke GeoJSON.


,kode_prov,nama_bps,provinsi,nama_peta,geojson_id,geojson_kode_prov
0,11,ACEH,Aceh,Aceh,11,11
1,12,SUMATERA UTARA,Sumatera Utara,Sumatera Utara,12,12
2,13,SUMATERA BARAT,Sumatera Barat,Sumatera Barat,13,13
3,14,RIAU,Riau,Riau,14,14
4,15,JAMBI,Jambi,Jambi,15,15


## 4. Membangun Tabel Final `provinsi_2025`
Menambahkan kolom wajib:
* `tahun`: 2025
* `kelompok`:
  - `<50%`: Kategori Merah (Rawan KLB)
  - `50-70%`: Kategori Kuning (Waspada / di bawah rata-rata)
  - `>=70%`: Kategori Hijau (Mendekati aman)
* `peringkat`: Peringkat 1 s.d. 38 (Peringkat 1 = tertinggi, Peringkat 38 = terendah)


In [4]:
df_final = df_bps.copy()
df_final['kode_prov'] = df_final['nama_bps'].map(lambda x: bps_to_meta[x][0])
df_final['provinsi'] = df_final['nama_bps'].map(lambda x: bps_to_meta[x][1])
df_final['tahun'] = 2025

def beri_kelompok(val):
    if val < 50.0:
        return '<50%'
    elif val < 70.0:
        return '50-70%'
    else:
        return '>=70%'

df_final['kelompok'] = df_final['imunisasi_campak_pct'].apply(beri_kelompok)
df_final['peringkat'] = df_final['imunisasi_campak_pct'].rank(ascending=False, method='min').astype(int)

# Urutkan berdasarkan peringkat (tertinggi ke terendah)
kolom_output = ['kode_prov', 'provinsi', 'tahun', 'imunisasi_campak_pct', 'kelompok', 'peringkat']
df_final = df_final[kolom_output].sort_values('peringkat').reset_index(drop=True)

print("5 Provinsi Tertinggi:")
display(df_final.head(5))

print("\n10 Provinsi Terendah:")
display(df_final.tail(10))


5 Provinsi Tertinggi:


,kode_prov,provinsi,tahun,imunisasi_campak_pct,kelompok,peringkat
0,34,DI Yogyakarta,2025,83.42,>=70%,1
1,51,Bali,2025,79.62,>=70%,2
2,31,DKI Jakarta,2025,73.61,>=70%,3
3,33,Jawa Tengah,2025,72.61,>=70%,4
4,53,Nusa Tenggara Timur,2025,72.54,>=70%,5



10 Provinsi Terendah:


,kode_prov,provinsi,tahun,imunisasi_campak_pct,kelompok,peringkat
28,15,Jambi,2025,59.16,50-70%,29
29,61,Kalimantan Barat,2025,58.84,50-70%,30
30,76,Sulawesi Barat,2025,57.79,50-70%,31
31,36,Banten,2025,55.35,50-70%,32
32,12,Sumatera Utara,2025,53.25,50-70%,33
33,14,Riau,2025,51.78,50-70%,34
34,94,Papua Tengah,2025,48.91,<50%,35
35,13,Sumatera Barat,2025,43.50,<50%,36
36,95,Papua Pegunungan,2025,36.69,<50%,37
37,11,Aceh,2025,27.79,<50%,38


## 5. Pengujian Kualitas Data (Assertion Checks)
Menguji data terhadap angka pembanding resmi yang disepakati kelompok:
* Jumlah provinsi = 38
* Rata-rata nasional = 63,52%
* Provinsi tertinggi = DI Yogyakarta (83,42%)
* Provinsi terendah = Aceh (27,79%)
* Jumlah provinsi < 50% = tepat 4 provinsi
* Jumlah provinsi di bawah angka nasional = 16 provinsi


In [5]:
assert len(df_final) == 38, "Jumlah baris harus 38 provinsi"
assert angka_nasional == 63.52, f"Nasional harus 63.52, diperoleh {angka_nasional}"

diy = df_final.loc[df_final['provinsi'] == 'DI Yogyakarta', 'imunisasi_campak_pct'].values[0]
assert diy == 83.42, f"DIY harus 83.42, diperoleh {diy}"

aceh = df_final.loc[df_final['provinsi'] == 'Aceh', 'imunisasi_campak_pct'].values[0]
assert aceh == 27.79, f"Aceh harus 27.79, diperoleh {aceh}"

jml_kritis = (df_final['kelompok'] == '<50%').sum()
assert jml_kritis == 4, f"Jumlah provinsi <50% harus 4, diperoleh {jml_kritis}"

di_bawah_nasional = (df_final['imunisasi_campak_pct'] < angka_nasional).sum()
assert di_bawah_nasional == 16, f"Jumlah provinsi < nasional harus 16, diperoleh {di_bawah_nasional}"

print(" SEMUA PENGECEKAN LOLOS (100% VALID & COCOK DENGAN DOKUMEN ANGKAKUNCI)")


 SEMUA PENGECEKAN LOLOS (100% VALID & COCOK DENGAN DOKUMEN ANGKAKUNCI)


## 6. Pratinjau Visualisasi untuk Looker Studio (Bagian ④)
Menampilkan grafik batang 10 provinsi dengan cakupan imunisasi terendah sebagai acuan desain grafik di Looker Studio.


In [6]:
sepuluh_terendah = df_final.sort_values('imunisasi_campak_pct').head(10)

plt.figure(figsize=(9, 5), dpi=150)
warna = ['#DC2626' if k == '<50%' else '#D97706' for k in sepuluh_terendah['kelompok']]

bars = plt.barh(sepuluh_terendah['provinsi'], sepuluh_terendah['imunisasi_campak_pct'], color=warna, height=0.65)
plt.axvline(angka_nasional, color='#1E293B', linestyle='--', linewidth=1.2, label=f'Rata-rata Nasional ({angka_nasional}%)')
plt.axvline(50, color='#DC2626', linestyle=':', linewidth=1.0, label='Batas Kritis (<50%)')

for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.8, bar.get_y() + bar.get_height()/2, f"{w:.1f}%", va='center', fontsize=9, fontweight='bold', color='#1E293B')

plt.title('10 Provinsi dengan Cakupan Imunisasi Campak Terendah (BPS 2025)', fontsize=12, fontweight='bold', pad=15)
plt.xlabel('Persentase Balita Pernah Diimunisasi Campak (%)', fontsize=10, fontweight='bold')
plt.xlim(0, 100)
plt.gca().invert_yaxis()
plt.legend(loc='lower right', frameon=True)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()

# Simpan pratinjau gambar
file_gambar = CLEAN / "pratinjau_10_provinsi_terendah.png"
plt.savefig(file_gambar, dpi=150)
plt.close()
print("Gambar pratinjau tersimpan di:", file_gambar)


Gambar pratinjau tersimpan di: ..\data\clean\pratinjau_10_provinsi_terendah.png


## 7. Menyimpan File Hasil Pembersihan
1. `provinsi_2025.csv`: Diunggah ke Google Sheets bersama pada tab `provinsi_2025`.
2. `tabel_padanan_provinsi.csv`: Diunggah ke tab `padanan_provinsi` atau disimpan di repo GitHub.


In [7]:
file_out_prov = CLEAN / "provinsi_2025.csv"
file_out_padanan = CLEAN / "tabel_padanan_provinsi.csv"

df_final.to_csv(file_out_prov, index=False)
df_padanan.to_csv(file_out_padanan, index=False)

print("Tersimpan dengan sukses:")
print(" -", file_out_prov)
print(" -", file_out_padanan)


Tersimpan dengan sukses:
 - ..\data\clean\provinsi_2025.csv
 - ..\data\clean\tabel_padanan_provinsi.csv


## 8. Ringkasan 3 Kalimat Temuan Utama (Untuk M1 & M3)
Tiga kalimat berikut diserahkan kepada penanggung jawab Makalah (**M3** untuk Bab III.b dan **M1** untuk Bab IV Kesimpulan & PPT):

1. **Disparitas Ekstrem:** Terdapat jurang ketimpangan imunisasi campak yang sangat tajam antarprovinsi di Indonesia pada 2025, dengan rentang selisih mencapai **55,63 poin persen** antara provinsi tertinggi (DI Yogyakarta: 83,42%) dan terendah (Aceh: 27,79%).
2. **Empat Kantong Kritis:** Sebanyak **4 dari 38 provinsi memiliki cakupan di bawah 50%** (Aceh 27,79%, Papua Pegunungan 36,69%, Sumatera Barat 43,50%, dan Papua Tengah 48,91%), yang menjadi kantong populasi paling rentan terjadinya Kejadian Luar Biasa (KLB).
3. **Mayoritas Belum Aman:** Sebanyak **16 provinsi berada di bawah rata-rata nasional (63,52%)**, termasuk provinsi berpopulasi padat seperti Jawa Barat (61,80%) dan Banten (55,35%), membuktikan bahwa risiko penularan campak tersebar luas di berbagai pulau.
